## Conexão e inspeção dos códigos de Q007 e Q023
- Antes de agregar, verificamos quais categorias distintas existem nas colunas q007 e q023 da tabela participantes (e conferimos o dicionário oficial na pasta dados/ caso queira ver o texto exato de cada letra).

In [1]:
import sys
sys.path.append("../src")

from pathlib import Path
import pandas as pd
from database import get_engine

engine = get_engine()

# 1. Confere a distribuição bruta de Q007 e Q023 no PostgreSQL
df_dist_q007 = pd.read_sql("""
    SELECT q007, COUNT(*) AS qtd
    FROM participantes
    GROUP BY q007
    ORDER BY q007;
""", engine)

df_dist_q023 = pd.read_sql("""
    SELECT q023, COUNT(*) AS qtd
    FROM participantes
    GROUP BY q023
    ORDER BY q023;
""", engine)

print("Categorias encontradas em Q007:")
display(df_dist_q007)

print("Categorias encontradas em Q023:")
display(df_dist_q023)

Categorias encontradas em Q007:


,q007,qtd
0,A,479899
1,B,1644626
2,C,866020
3,D,434003
4,E,202017
5,F,233062
6,G,271843
7,H,137918
8,I,106469
9,J,85387


Categorias encontradas em Q023:


,q023,qtd
0,A,3693571
1,B,176099
2,C,87913
3,D,659592
4,E,173102
5,F,20495


## Consulta SQL com CTEs cruzando perfil socioeconômico e notas por município

Usamos duas CTEs (WITH) no PostgreSQL para que o banco faça toda a agregação pesada antes de enviar o DataFrame final para o Pandas:

    socio_municipio: calcula o total de inscritos por município e a proporção (%) de cada faixa de q007 e q023.

    notas_municipio: calcula a média das 4 provas objetivas (media_objetivas), a média da redação (media_redacao) e a média geral apenas para candidatos presentes nas 4 provas objetivas e com redação válida.

    Filtramos municípios com pelo menos 30 participantes presentes (HAVING COUNT(*) >= 30) para evitar distorções estatísticas de cidades muito pequenas.

In [4]:
from sqlalchemy import text

query_uf = text("""
WITH socio_uf AS (
    SELECT
        m.co_uf,
        m.sg_uf,
        COUNT(*) AS total_inscritos,
        
        -- Proporcao percentual das respostas de Q007 (Renda)
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q007 = 'A') / COUNT(*), 2) AS pct_q007_a,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q007 = 'B') / COUNT(*), 2) AS pct_q007_b,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q007 = 'C') / COUNT(*), 2) AS pct_q007_c,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q007 IN ('D', 'E', 'F', 'G')) / COUNT(*), 2) AS pct_q007_d_a_g,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q007 >= 'H') / COUNT(*), 2) AS pct_q007_h_ou_mais,
        
        -- Proporcao percentual das respostas de Q023 (Tipo de escola no Ensino Medio)
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'A') / COUNT(*), 2) AS pct_q023_a,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'B') / COUNT(*), 2) AS pct_q023_b,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'C') / COUNT(*), 2) AS pct_q023_c,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'D') / COUNT(*), 2) AS pct_q023_d,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'E') / COUNT(*), 2) AS pct_q023_e,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q023 = 'F') / COUNT(*), 2) AS pct_q023_f
    FROM participantes p
    JOIN municipios m ON p.co_municipio_prova = m.co_municipio
    GROUP BY m.co_uf, m.sg_uf
),
notas_uf AS (
    SELECT
        m.co_uf,
        m.sg_uf,
        COUNT(*) AS total_presentes_4_provas,
        ROUND(AVG(r.nu_nota_cn), 2) AS media_cn,
        ROUND(AVG(r.nu_nota_ch), 2) AS media_ch,
        ROUND(AVG(r.nu_nota_lc), 2) AS media_lc,
        ROUND(AVG(r.nu_nota_mt), 2) AS media_mt,
        ROUND(AVG((r.nu_nota_cn + r.nu_nota_ch + r.nu_nota_lc + r.nu_nota_mt) / 4.0), 2) AS media_objetivas,
        ROUND(AVG(r.nu_nota_redacao), 2) AS media_redacao,
        ROUND(AVG((r.nu_nota_cn + r.nu_nota_ch + r.nu_nota_lc + r.nu_nota_mt + r.nu_nota_redacao) / 5.0), 2) AS media_geral
    FROM resultados r
    JOIN municipios m ON r.co_municipio_prova = m.co_municipio
    WHERE r.tp_presenca_cn = 1
      AND r.tp_presenca_ch = 1
      AND r.tp_presenca_lc = 1
      AND r.tp_presenca_mt = 1
      AND r.nu_nota_redacao IS NOT NULL
    GROUP BY m.co_uf, m.sg_uf
)
SELECT
    s.co_uf,
    s.sg_uf,
    s.total_inscritos,
    n.total_presentes_4_provas,
    -- Indicadores Q007
    s.pct_q007_a,
    s.pct_q007_b,
    s.pct_q007_c,
    s.pct_q007_d_a_g,
    s.pct_q007_h_ou_mais,
    -- Indicadores Q023
    s.pct_q023_a,
    s.pct_q023_b,
    s.pct_q023_c,
    s.pct_q023_d,
    s.pct_q023_e,
    s.pct_q023_f,
    -- Desempenho
    n.media_cn,
    n.media_ch,
    n.media_lc,
    n.media_mt,
    n.media_objetivas,
    n.media_redacao,
    n.media_geral
FROM socio_uf s
JOIN notas_uf n ON s.co_uf = n.co_uf
ORDER BY n.media_geral DESC;
""")

df_analise_uf = pd.read_sql(query_uf, engine)

print(f"Total de estados analisados: {len(df_analise_uf)}")
display(df_analise_uf)

Total de estados analisados: 27


,co_uf,sg_uf,total_inscritos,total_presentes_4_provas,pct_q007_a,pct_q007_b,pct_q007_c,pct_q007_d_a_g,pct_q007_h_ou_mais,pct_q023_a,...,pct_q023_d,pct_q023_e,pct_q023_f,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
0,35,SP,751612,511757,3.59,17.40,19.73,35.45,23.82,67.19,...,20.56,6.17,0.32,517.61,537.25,558.88,560.49,543.56,615.17,557.88
1,53,DF,82955,56633,5.42,25.52,16.90,22.82,29.34,67.01,...,21.18,4.49,0.76,519.86,538.54,557.99,548.08,541.12,619.47,556.79
2,42,SC,110459,72924,2.35,12.29,17.46,40.25,27.64,70.73,...,18.53,3.86,0.44,517.30,537.52,555.70,553.59,541.03,614.68,555.76
3,31,MG,464937,307991,5.27,26.78,21.16,30.78,16.01,78.61,...,13.08,3.18,0.39,512.25,530.90,547.63,548.29,534.77,628.85,553.59
4,43,RS,186503,121437,4.30,21.04,20.68,34.55,19.43,76.59,...,13.48,3.53,0.64,510.77,534.31,555.30,549.20,537.40,613.73,552.66
5,32,ES,85910,56380,4.50,25.31,22.28,31.27,16.64,80.18,...,11.80,3.18,0.36,512.82,529.51,545.43,545.16,533.23,611.51,548.89
6,33,RJ,328943,221151,7.70,30.41,19.59,24.78,17.52,62.90,...,21.22,6.72,0.51,508.75,529.01,549.81,537.75,531.33,613.01,547.66
7,41,PR,195836,132004,3.12,16.59,19.79,38.08,22.42,74.63,...,15.98,3.04,0.39,511.22,528.89,548.93,544.74,533.44,590.68,544.89
8,52,GO,166747,109987,5.40,29.31,22.37,27.83,15.09,76.79,...,13.54,2.52,0.52,502.51,514.83,534.38,521.96,518.42,621.58,539.05
9,24,RN,113208,81832,12.23,43.70,16.32,17.53,10.23,78.43,...,13.85,2.49,0.26,505.54,513.39,534.98,515.64,517.39,595.89,533.09


## DataFrame de Correlação (df_correlacao) e salvamento

- Aqui calculamos a correlação de Pearson (relação linear) e de Spearman (relação monotônica, mais robusta a assimetrias) entre as variáveis de Q007/Q023 e as notas (media_objetivas, media_redacao e media_geral), salvando os resultados em DataFrames prontos para a próxima etapa da sua análise.

In [5]:
colunas_socio = [
    "pct_q007_a", "pct_q007_b", "pct_q007_c", "pct_q007_d_a_g", "pct_q007_h_ou_mais",
    "pct_q023_a", "pct_q023_b", "pct_q023_c", "pct_q023_d", "pct_q023_e", "pct_q023_f"
]
colunas_notas = ["media_cn", "media_ch", "media_lc", "media_mt", "media_objetivas", "media_redacao", "media_geral"]

df_correlacao_pearson = (
    df_analise_uf[colunas_socio + colunas_notas]
    .corr(method="pearson")
    .loc[colunas_socio, colunas_notas]
    .round(4)
)

df_correlacao_spearman = (
    df_analise_uf[colunas_socio + colunas_notas]
    .corr(method="spearman")
    .loc[colunas_socio, colunas_notas]
    .round(4)
)

print("Correlacao de Pearson por Estado (Socioeconomico x Desempenho):")
display(df_correlacao_pearson)

print("Correlacao de Spearman por Estado (Socioeconomico x Desempenho):")
display(df_correlacao_spearman)

df_analise_uf.to_csv("../dados/df_analise_uf_q007_q023.csv", index=False)
print("DataFrame salvo em ../dados/df_analise_uf_q007_q023.csv")

Correlacao de Pearson por Estado (Socioeconomico x Desempenho):


,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
pct_q007_a,-0.7858,-0.8410,-0.8118,-0.7908,-0.8134,-0.6310,-0.7829
pct_q007_b,-0.7850,-0.8462,-0.8202,-0.8043,-0.8214,-0.6253,-0.7872
pct_q007_c,0.4920,0.5486,0.5098,0.5177,0.5231,0.4085,0.5042
pct_q007_d_a_g,0.7415,0.8006,0.7734,0.7740,0.7812,0.5939,0.7485
pct_q007_h_ou_mais,0.8395,0.8914,0.8733,0.8249,0.8612,0.6621,0.8272
pct_q023_a,-0.8051,-0.7683,-0.8028,-0.7967,-0.8008,-0.7651,-0.8112
pct_q023_b,0.5461,0.4750,0.5137,0.5000,0.5106,0.5511,0.5351
pct_q023_c,0.6182,0.6254,0.6537,0.6395,0.6417,0.5849,0.6421
pct_q023_d,0.8284,0.7856,0.8177,0.8139,0.8188,0.7794,0.8286
pct_q023_e,0.6891,0.6885,0.7216,0.7209,0.7148,0.6650,0.7190


Correlacao de Spearman por Estado (Socioeconomico x Desempenho):


,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
pct_q007_a,-0.7985,-0.8455,-0.8144,-0.7985,-0.8199,-0.6325,-0.7712
pct_q007_b,-0.7863,-0.8120,-0.8016,-0.7473,-0.7973,-0.5763,-0.7489
pct_q007_c,0.4860,0.5372,0.4768,0.4652,0.5018,0.3303,0.4439
pct_q007_d_a_g,0.7485,0.7821,0.7595,0.7228,0.7582,0.5659,0.7162
pct_q007_h_ou_mais,0.8370,0.8828,0.8632,0.7924,0.8443,0.6508,0.8100
pct_q023_a,-0.8150,-0.7344,-0.8175,-0.8065,-0.8181,-0.7973,-0.8142
pct_q023_b,0.5679,0.4931,0.5652,0.5139,0.5652,0.5118,0.5363
pct_q023_c,0.6581,0.6001,0.6667,0.6593,0.6832,0.6386,0.6881
pct_q023_d,0.8010,0.7045,0.7937,0.7747,0.7900,0.7564,0.7794
pct_q023_e,0.7581,0.6659,0.7342,0.7687,0.7590,0.7489,0.7976


DataFrame salvo em ../dados/df_analise_uf_q007_q023.csv


## Verificar categorias de Q006 no banco e no dicionário do INEP

In [6]:
from sqlalchemy import text
from database import get_engine

engine = get_engine()

# 1. Verifica as categorias distintas de Q006 na tabela participantes
df_dist_q006 = pd.read_sql(text("""
    SELECT
        q006,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM participantes
    GROUP BY q006
    ORDER BY q006;
"""), engine)

print("Distribuição nacional de Q006:")
display(df_dist_q006)

# 2. Busca a descrição oficial de Q006 no arquivo de dicionário (se disponível em dados/)
dicionarios = list(Path("../dados").rglob("*DICIONARIO*"))
if dicionarios:
    df_dic = pd.read_excel(dicionarios[0], sheet_name="PARTICIPANTES_2025")
    display(df_dic[df_dic.iloc[:, 0].astype(str).str.contains("Q006", na=False)])

Distribuição nacional de Q006:


,q006,qtd,percentual
0,A,3454810,71.81
1,B,1355962,28.19


## Consulta SQL agregada por Estado (UF) cruzando Q006 e notas

- Assim como fizemos para Q007 e Q023, agrupamos os percentuais de cada resposta de q006 por UF e cruzamos com as médias das provas objetivas, redação e média geral (considerando apenas candidatos presentes nas 4 provas objetivas e com redação válida).

- (Nota: o SQL abaixo contempla as alternativas A a E mais comuns para situação de trabalho; caso a Célula 1 mostre alguma letra adicional em Q006, basta acrescentar uma linha seguindo o mesmo padrão).

In [7]:
query_uf_q006 = text("""
WITH trabalho_uf AS (
    SELECT
        m.co_uf,
        m.sg_uf,
        COUNT(*) AS total_inscritos,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q006 = 'A') / COUNT(*), 2) AS pct_q006_a,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q006 = 'B') / COUNT(*), 2) AS pct_q006_b,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q006 = 'C') / COUNT(*), 2) AS pct_q006_c,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q006 = 'D') / COUNT(*), 2) AS pct_q006_d,
        ROUND(100.0 * COUNT(*) FILTER (WHERE p.q006 = 'E') / COUNT(*), 2) AS pct_q006_e
    FROM participantes p
    JOIN municipios m ON p.co_municipio_prova = m.co_municipio
    WHERE p.q006 IS NOT NULL
    GROUP BY m.co_uf, m.sg_uf
),
notas_uf AS (
    SELECT
        m.co_uf,
        m.sg_uf,
        COUNT(*) AS total_presentes_4_provas,
        ROUND(AVG(r.nu_nota_cn), 2) AS media_cn,
        ROUND(AVG(r.nu_nota_ch), 2) AS media_ch,
        ROUND(AVG(r.nu_nota_lc), 2) AS media_lc,
        ROUND(AVG(r.nu_nota_mt), 2) AS media_mt,
        ROUND(AVG((r.nu_nota_cn + r.nu_nota_ch + r.nu_nota_lc + r.nu_nota_mt) / 4.0), 2) AS media_objetivas,
        ROUND(AVG(r.nu_nota_redacao), 2) AS media_redacao,
        ROUND(AVG((r.nu_nota_cn + r.nu_nota_ch + r.nu_nota_lc + r.nu_nota_mt + r.nu_nota_redacao) / 5.0), 2) AS media_geral
    FROM resultados r
    JOIN municipios m ON r.co_municipio_prova = m.co_municipio
    WHERE r.tp_presenca_cn = 1
      AND r.tp_presenca_ch = 1
      AND r.tp_presenca_lc = 1
      AND r.tp_presenca_mt = 1
      AND r.nu_nota_redacao IS NOT NULL
    GROUP BY m.co_uf, m.sg_uf
)
SELECT
    t.co_uf,
    t.sg_uf,
    t.total_inscritos,
    n.total_presentes_4_provas,
    t.pct_q006_a,
    t.pct_q006_b,
    t.pct_q006_c,
    t.pct_q006_d,
    t.pct_q006_e,
    n.media_cn,
    n.media_ch,
    n.media_lc,
    n.media_mt,
    n.media_objetivas,
    n.media_redacao,
    n.media_geral
FROM trabalho_uf t
JOIN notas_uf n ON t.co_uf = n.co_uf
ORDER BY n.media_geral DESC;
""")

df_analise_uf_q006 = pd.read_sql(query_uf_q006, engine)

print(f"Total de estados analisados: {len(df_analise_uf_q006)}")
display(df_analise_uf_q006)

Total de estados analisados: 27


,co_uf,sg_uf,total_inscritos,total_presentes_4_provas,pct_q006_a,pct_q006_b,pct_q006_c,pct_q006_d,pct_q006_e,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
0,35,SP,751612,511757,69.31,30.69,0.0,0.0,0.0,517.61,537.25,558.88,560.49,543.56,615.17,557.88
1,53,DF,82955,56633,70.97,29.03,0.0,0.0,0.0,519.86,538.54,557.99,548.08,541.12,619.47,556.79
2,42,SC,110459,72924,59.66,40.34,0.0,0.0,0.0,517.30,537.52,555.70,553.59,541.03,614.68,555.76
3,31,MG,464937,307991,67.87,32.13,0.0,0.0,0.0,512.25,530.90,547.63,548.29,534.77,628.85,553.59
4,43,RS,186503,121437,59.72,40.28,0.0,0.0,0.0,510.77,534.31,555.30,549.20,537.40,613.73,552.66
5,32,ES,85910,56380,63.67,36.33,0.0,0.0,0.0,512.82,529.51,545.43,545.16,533.23,611.51,548.89
6,33,RJ,328943,221151,73.16,26.84,0.0,0.0,0.0,508.75,529.01,549.81,537.75,531.33,613.01,547.66
7,41,PR,195836,132004,65.27,34.73,0.0,0.0,0.0,511.22,528.89,548.93,544.74,533.44,590.68,544.89
8,52,GO,166747,109987,69.85,30.15,0.0,0.0,0.0,502.51,514.83,534.38,521.96,518.42,621.58,539.05
9,24,RN,113208,81832,71.96,28.04,0.0,0.0,0.0,505.54,513.39,534.98,515.64,517.39,595.89,533.09


## DataFrames de Correlação (Q006 x Notas) e salvamento

- Filtra dinamicamente apenas as colunas pct_q006_* que possuem variação (desvio padrão maior que zero, caso alguma letra não exista na edição 2025) e gera os DataFrames de correlação de Pearson e Spearman:

In [8]:
colunas_q006 = [
    col for col in ["pct_q006_a", "pct_q006_b", "pct_q006_c", "pct_q006_d", "pct_q006_e"]
    if df_analise_uf_q006[col].std() > 0
]
colunas_notas = ["media_cn", "media_ch", "media_lc", "media_mt", "media_objetivas", "media_redacao", "media_geral"]

df_corr_q006_pearson = (
    df_analise_uf_q006[colunas_q006 + colunas_notas]
    .corr(method="pearson")
    .loc[colunas_q006, colunas_notas]
    .round(4)
)

df_corr_q006_spearman = (
    df_analise_uf_q006[colunas_q006 + colunas_notas]
    .corr(method="spearman")
    .loc[colunas_q006, colunas_notas]
    .round(4)
)

print("Correlação de Pearson por Estado (Q006 x Desempenho):")
display(df_corr_q006_pearson)

print("Correlação de Spearman por Estado (Q006 x Desempenho):")
display(df_corr_q006_spearman)

df_analise_uf_q006.to_csv("../dados/df_analise_uf_q006.csv", index=False)
print("DataFrame salvo em ../dados/df_analise_uf_q006.csv")

Correlação de Pearson por Estado (Q006 x Desempenho):


,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
pct_q006_a,-0.6453,-0.7187,-0.679,-0.6477,-0.6764,-0.5098,-0.6468
pct_q006_b,0.6453,0.7187,0.679,0.6477,0.6764,0.5098,0.6468


Correlação de Spearman por Estado (Q006 x Desempenho):


,media_cn,media_ch,media_lc,media_mt,media_objetivas,media_redacao,media_geral
pct_q006_a,-0.6585,-0.7184,-0.6616,-0.6155,-0.6628,-0.4995,-0.6227
pct_q006_b,0.6585,0.7184,0.6616,0.6155,0.6628,0.4995,0.6227


DataFrame salvo em ../dados/df_analise_uf_q006.csv
